# 4D SfM — DEM + DoD + M3C2 raster (monthly batch)

Runs the full per-date workflow for one hand-picked date per month, producing
DEM + orthoimage + DoD + stable-terrain DoD + M3C2 raster (with histograms)
for each. All logic lives in `tlapse4d.pipeline_4dsfm`; this notebook only says
*which glacier*, *which knobs*, and *which dates*.

Reference rasters (`reference_dem.tif`, `reference_ortho.tif`,
`reference_dem_stable.tif`) cache in `_ref_cache/` — built on the
first iteration, skipped on every subsequent date.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ["AGISOFT_LICENSE_PATH"] = "/home/asus/.config/Agisoft/license.lic"

import Metashape  # noqa: F401  — must import after AGISOFT_LICENSE_PATH is set
from tlapse4d.pipeline_4dsfm import run_batch

## Configuration

Paths (*which glacier*) come from `site_config*.py`; pipeline knobs (*how it is
run*) come from `run_config.py`. This run overrides five of them — including
setting `time_window` and `exclude_cameras` back to `None`, which the shared
defaults set for the full-record batch but which this monthly run has never used.

In [ ]:
import site_config_north as site
from run_config import params

params = params | dict(
    m_sp2p_max_disp = 1,      # tighter Stage 3 ICP than the full-record batch
    max_unaligned   = 10,     # looser cloud-cover gate
    verbose         = True,
    time_window     = None,   # no daytime filter on this run
    exclude_cameras = None,   # no camera exclusions on this run
)

# ── Dates to process (one per month) ─────────────────────────────────
monthly_dates = [
    "2024-01-18",
    "2024-02-18",
    "2024-03-17",
    "2024-04-19",
    "2024-05-17",
    "2024-06-15",
    "2024-07-01",
    "2024-08-17",
    "2024-09-18",
    "2024-10-18",
    "2024-11-15",
]

## Run

`run_batch` runs each date in turn, records a failure as a row instead of
stopping, prints a summary and writes `output/batch_summary.csv`.

In [ ]:
df = run_batch(
    monthly_dates,
    tlcam_dir    = site.tlcam_dir,
    ref_cloud    = site.ref_cloud,
    glacier_mask = site.glacier_mask,
    registry_csv = site.registry_csv,
    output_dir   = site.output_dir,
    **params,
)
df